# Topic similarity

In [10]:
import evaluate
import ir_datasets

from topic_gen.models import TRECTopic, Topics
from topic_gen.evaluate import embedding_cosine_similarity
from dotenv import load_dotenv
import json
from pydantic_core import from_json
load_dotenv()

True

In [11]:
topics = Topics[TRECTopic].from_jsonl("../data/processed/trec-reference/topics-robust-gemini-flash-trec-base-q4-d1.jsonl")

In [ ]:
with open("../data/processed/trec-reference/topics-robust-gemini-flash-trec-base-q4-d1.jsonl", "r") as f:
    topics_json = [from_json(TRECTopic.model_json_schema(), line) for line in f]

In [ ]:
evaluate = evaluate_similarity(generated_topics, reference_topics)

In [10]:
class evaluate_similarity:
    def __init__(self, reference_topics, candidate_topics):
        self.meteor_metric = evaluate.load("meteor")
        self.bertscore_metric = evaluate.load("bertscore")
        self.cosine_metric = embedding_cosine_similarity()

        self.data = self.load_data(reference_topics, candidate_topics)

    def load_data(self, reference_topics, candidate_topics):
        fields = reference_topics[0]._fields  # infer available fields from first reference topic
        fields = [field for field in fields if field not in ["query_id", "id"]]  # exclude ID fields
        
        similarity = {}
        for field in fields:
            similarity[field] = {}
            similarity[field]["candidate"] = []
            similarity[field]["reference"] = []
        
        for reference, candidate in zip(reference_topics, candidate_topics):
            # assert reference._fields == candidate.dict().keys(), 
            # assert reference.query_id == candidate.id, "Topic IDs between the candidate and reference topics do not match. "
            
            for field in fields:
                similarity[field]["candidate"].append(candidate.__getattribute__(field))
                similarity[field]["reference"].append(reference.__getattribute__(field))
        
        return similarity
    
    def compute(self):
        results = {}
        for field in self.data.keys():
            meteor = [None for _ in range(len(sim["title"]["candidate"]))]
            meteor_average = self.meteor_metric.compute(
                    predictions=self.data[field]["candidate"],
                    references=self.data[field]["reference"]
                )

            bertscore = self.bertscore_metric.compute(
                predictions=self.data[field]["candidate"],
                references=self.data[field]["reference"],
                model_type="distilbert-base-uncased"
            )
            bertscore_precision = bertscore["precision"]
            bertscore_recall = bertscore["recall"]
            bertscore_f1 = bertscore["f1"]
            
            cosine = self.cosine_metric.compute(
                    reference_topics=self.data[field]["reference"],
                    candidate_topics=self.data[field]["candidate"]
                )
            res = pd.DataFrame(results)

            results[field] = {
                "meteor": meteor,
                "meteor_average": meteor_average,
                "bertscore_precision": bertscore_precision,
                "bertscore_recall": bertscore_recall,
                "bertscore_f1": bertscore_f1,
                "cosine": cosine
            }
        
        return results

In [122]:
bertscore_results

{'precision': [0.8225515484809875,
  0.9555574655532837,
  0.8526989817619324,
  0.7899551391601562,
  0.9677736759185791,
  0.9360541105270386,
  0.8516387939453125,
  0.711989164352417,
  0.9063488245010376],
 'recall': [0.7838496565818787,
  0.9555574655532837,
  0.7776868939399719,
  0.7654914855957031,
  0.9677736759185791,
  0.9360541105270386,
  0.8743951320648193,
  0.7145860195159912,
  0.8816832304000854],
 'f1': [0.802734375,
  0.9555574655532837,
  0.8134673833847046,
  0.7775309681892395,
  0.9677736759185791,
  0.9360541105270386,
  0.862866997718811,
  0.7132852077484131,
  0.8938459157943726],
 'hashcode': 'distilbert-base-uncased_L5_no-idf_version=0.3.12(hug_trans=4.53.2)'}

In [ ]:
np.arange(len(sim["title"]["candidate"]), )

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [ ]:
meteor_metric = evaluate.load("meteor")

meteor_results = meteor_metric.compute(
        predictions=sim["title"]["candidate"], references=sim["title"]["reference"]
    )

In [ ]:
bertscore_metric = evaluate.load("bertscore")

bertscore_results = bertscore_metric.compute(
    predictions=sim["title"]["candidate"],
    references=sim["title"]["reference"],
    model_type="distilbert-base-uncased"
)

In [103]:
cosine_metric = llm_cosine_similarity()
cosine_results = cosine_metric.compute(
    reference_topics=sim["title"]["reference"],
    candidate_topics=sim["title"]["candidate"]
)

In [ ]:
# load data
candidate_topics = Topics.read_xml("test-topics.xml", TRECTopic)
dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")
reference_topics = [topic for topic in dataset.queries_iter()][1:]

evaluate = evaluate_similarity(reference_topics, candidate_topics)
results = evaluate.compute()

[nltk_data] Downloading package wordnet to /Users/jueri/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/jueri/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/jueri/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [113]:
results.keys()

dict_keys(['title', 'description', 'narrative'])

In [116]:
results["title"].keys()

dict_keys(['meteor', 'bertscore', 'cosine'])

In [119]:
results["title"]["cosine"]

[np.float64(0.72072856910086),
 np.float64(0.7385319330041583),
 np.float64(0.8442269040091606),
 np.float64(0.6120470608667965),
 np.float64(0.970294654933186),
 np.float64(0.782885408784191),
 np.float64(0.7311912769913711),
 np.float64(0.3955435735980768),
 np.float64(0.8748898211685361)]